<a href="https://colab.research.google.com/github/meghanavanamala/predictiveanalysis/blob/recommand-system-to-suggest-course/recommand_system_to_suggest_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

# -----------------------------
# 1. Sample Student-Course Data
# -----------------------------
data = [
    {"student": "Alice", "courses": ["Python", "Data Science"], "interests": ["AI", "ML"], "focus": "Data Science", "performance": 85},
    {"student": "Bob", "courses": ["HTML", "CSS", "JavaScript"], "interests": ["Web Dev"], "focus": "Web Development", "performance": 70},
    {"student": "Charlie", "courses": ["Python", "ML", "Deep Learning"], "interests": ["AI", "Robotics"], "focus": "AI", "performance": 90},
    {"student": "David", "courses": ["SQL", "Data Analysis"], "interests": ["Data"], "focus": "Data Science", "performance": 75},
    {"student": "Eva", "courses": ["Java", "OOP", "System Design"], "interests": ["Backend"], "focus": "Software Engineering", "performance": 88},
]

df = pd.DataFrame(data)

# -----------------------------
# 2. Encode Interests & Focus
# -----------------------------
mlb = MultiLabelBinarizer()
interests_encoded = mlb.fit_transform(df['interests'])

# Append focus area as one-hot
focus_dummies = pd.get_dummies(df['focus'])

# Combine all features
features = pd.DataFrame(interests_encoded, columns=mlb.classes_)
features = pd.concat([features, focus_dummies, df[['performance']]], axis=1)

# -----------------------------
# 3. Calculate Similarity Matrix
# -----------------------------
similarity_matrix = cosine_similarity(features)

# -----------------------------
# 4. Recommend Courses
# -----------------------------
def recommend_courses(student_name):
    if student_name not in df['student'].values:
        return f"No student found with name '{student_name}'"

    idx = df[df['student'] == student_name].index[0]
    similar_scores = list(enumerate(similarity_matrix[idx]))

    # Sort students by similarity (excluding self)
    similar_scores = sorted(similar_scores, key=lambda x: x[1], reverse=True)[1:]

    # Get similar students' courses
    similar_students = [i for i, score in similar_scores[:3]]  # top 3 similar students
    course_suggestions = set()

    for i in similar_students:
        other_courses = df.iloc[i]['courses']
        for course in other_courses:
            if course not in df.iloc[idx]['courses']:
                course_suggestions.add(course)

    if not course_suggestions:
        return f"No new courses to recommend for {student_name}."

    return f"Recommended courses for {student_name}: {list(course_suggestions)}"

# -----------------------------
# 5. Example Usage
# -----------------------------
print(recommend_courses("David"))


Recommended courses for David: ['Python', 'ML', 'Deep Learning', 'System Design', 'OOP', 'Data Science', 'Java']
